# 뉴스 인사이트 – 실제 데이터 수집 (Colab)

1. 아래 셀을 **위에서부터 순서대로** 실행하세요.
2. 첫 셀에서 `news-insight.zip` 파일을 업로드합니다.
3. 마지막 셀이 `news-data.zip` 을 내려받게 해줍니다. 그 파일을 Claude 대화창에 올려주시면 실제 데이터로 데모 사이트를 만들어 드립니다.

수집 기간을 바꾸려면 2번 셀의 `START`, `END` 값을 고치세요. **7일 기준 약 10~20분** 걸립니다(기사 상세페이지 발행시각 확인 때문).

In [ ]:
# 1) 소스 업로드 및 설치
from google.colab import files
import os, zipfile, shutil
up = files.upload()  # news-insight.zip 선택
name = next(iter(up))
shutil.rmtree('/content/newsroom', ignore_errors=True)
zipfile.ZipFile(name).extractall('/content')
os.chdir('/content/newsroom')
!pip install -q -r collector/requirements.txt
print('✅ 준비 완료:', os.getcwd())

In [ ]:
# 2) 수집 기간 설정 (한국시간 기준, YYYY-MM-DD)
import datetime as dt
KST = dt.timezone(dt.timedelta(hours=9))
today = dt.datetime.now(KST).date()
END = (today - dt.timedelta(days=1)).isoformat()   # 어제
START = (today - dt.timedelta(days=7)).isoformat() # 7일 전
print('수집 기간:', START, '~', END)

In [ ]:
# 3) 랭킹 수집 (10개 매체, 빠름)
!python collector/collect.py --start $START --end $END --only ranking

In [ ]:
# 4) 전체 발행목록 수집 (7개 매체, 시간이 걸립니다)
!python collector/collect.py --start $START --end $END --only publish

In [ ]:
# 5) 검색 키워드 (구글 급상승 검색어 / 네이버 검색광고 API 키가 있으면 검색량도)
# 네이버 검색량을 넣으려면 아래 3줄의 주석을 풀고 키를 입력하세요.
# os.environ['NAVER_AD_API_KEY'] = ''
# os.environ['NAVER_AD_SECRET'] = ''
# os.environ['NAVER_AD_CUSTOMER_ID'] = ''
!python collector/search.py

In [ ]:
# (선택) 기존 엑셀을 가져온 경우: 기자명 채우기
# 엑셀 파일들을 먼저 업로드해서 가져온 뒤 실행하세요.
# up2 = files.upload()
# !python collector/import_excel.py *.xlsx
!python collector/fill_reporters.py

In [ ]:
# 6) 결과 확인 + 내려받기
import json
m = json.load(open('data/meta.json', encoding='utf-8'))
print('랭킹 수집일:', m.get('ranking_days'))
print('발행 수집일:', m.get('article_days'))
print('경고:', (m.get('last_run') or {}).get('errors'))
shutil.make_archive('/content/news-data', 'zip', '/content/newsroom', 'data')
files.download('/content/news-data.zip')